# SE4050 Deep Learning 2026 - Custom CNN Road-Damage Classification

**Task:** multi-label classification of whole road images into `D00`, `D10`,
`D20`, `D40` - the same prediction question the VGG16 and ResNet-50 experiments
ask, answered by a convolutional network built from scratch (no pretrained
weights).

Design rules (identical to the VGG16 protocol so the three multi-label models
are directly comparable):

* **data** - the shared leakage-safe split manifest produced by the VGG16
  pipeline (`results/IT22064868/metrics/split_manifest.csv`, 3005/645/645);
* **seed** - 42 for Python, NumPy and TensorFlow;
* **objective** - `binary_crossentropy` over 4 sigmoid outputs, Adam 1e-3;
* **callbacks** - early stopping and LR reduction on `val_auc`
  (patience 12 / 5), best-weights checkpoint;
* **evaluation** - test set touched only in section 6; threshold tuning on
  validation only; the full VGG16 metric suite (subset / hamming accuracy,
  per-class P/R/F1/ROC-AUC, micro + macro averages, per-source results).

Results are written to `results/IT23331518/` (metrics, figures, model).

In [1]:
import hashlib
import json
import os
import random
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("GPU available   :", tf.config.list_physical_devices("GPU"))
print("project root    :", ROOT)

TensorFlow version: 2.21.0


GPU available   : []
project root    : C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment


In [2]:
class Config:
    results_dir = ROOT / "results/IT23331518"
    metrics_dir = results_dir / "metrics"
    fig_dir = results_dir / "figures"
    shared_manifest = ROOT / "results/IT22064868/metrics/split_manifest.csv"
    manifest_copy = metrics_dir / "split_manifest.csv"
    vgg_run_meta = ROOT / "src/IT22064868/models/run_meta.json"

    checkpoint_path = results_dir / "custom_cnn_model.keras"
    weights_path = results_dir / "custom_cnn_weights.weights.h5"
    history_path = metrics_dir / "training_history.json"
    run_meta_path = metrics_dir / "run_meta.json"

    img_size = 224
    batch_size = 64
    epochs = 100
    lr = 1e-3
    seed = 42
    dropout = 0.5
    patience_earlystop = 12
    patience_lr = 5
    class_names = ["D00", "D10", "D20", "D40"]

    @classmethod
    def ensure_dirs(cls):
        for sub in ("metrics", "figures", "models"):
            (cls.results_dir / sub).mkdir(parents=True, exist_ok=True)

    @classmethod
    def as_dict(cls):
        return {
            "img_size": cls.img_size,
            "batch_size": cls.batch_size,
            "epochs": cls.epochs,
            "lr": cls.lr,
            "seed": cls.seed,
            "dropout": cls.dropout,
            "patience_earlystop": cls.patience_earlystop,
            "patience_lr": cls.patience_lr,
            "class_names": cls.class_names,
            "checkpoint_path": str(cls.checkpoint_path),
            "shared_manifest": str(cls.shared_manifest),
        }


Config.ensure_dirs()


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def save_json(obj, path):
    path = Path(path)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2)
    print("saved ->", path)


def load_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def hashlib_md5(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.elapsed = time.time() - self.t0


set_global_seed(Config.seed)
RUN = {}

## 2. Shared leakage-safe split

The manifest is produced by the VGG16 pipeline: 4,295 labelled images,
near-duplicate clusters assigned to train/val/test as whole units
(3005 / 645 / 645). This notebook **reuses it as-is** - no re-splitting - and
re-checks the leakage invariants before training. The split MD5 is verified
against the value recorded in VGG16's `run_meta.json`.

In [3]:
df = pd.read_csv(Config.shared_manifest)
df["split"] = df["split"].astype(str)
print(f"manifest rows: {len(df)}")
print(df["split"].value_counts().to_dict())


def split_md5(frame):
    """MD5 over train+val+test paths in manifest order (VGG16 convention)."""
    parts = []
    for sp in ("train", "val", "test"):
        parts += list(frame[frame["split"] == sp]["path"])
    return hashlib_md5("\n".join(parts))


MD5 = split_md5(df)
vgg_meta = load_json(Config.vgg_run_meta)
VGG_MD5 = vgg_meta["split_md5"]
print("split md5 (this run):", MD5)
print("split md5 (VGG16)   :", VGG_MD5)
if MD5 != VGG_MD5:
    raise RuntimeError("split manifest differs from the shared VGG16 split")
print("manifest matches the shared split - the three models are comparable.")

# leakage invariants (same checks the VGG16 notebook runs)
cluster_splits = df.groupby("cluster_id")["split"].nunique()
crossing = cluster_splits[cluster_splits > 1]
dup_paths = int(df["path"].duplicated().sum())
leak = {
    "clusters_spanning_multiple_splits": int(len(crossing)),
    "duplicate_paths": dup_paths,
    "split_counts": df["split"].value_counts().to_dict(),
    "passed": bool(len(crossing) == 0 and dup_paths == 0),
}
RUN["split_md5"] = MD5
RUN["leakage"] = leak
print("leakage report:", leak)
if not leak["passed"]:
    raise RuntimeError("leakage check failed - aborting")

df.to_csv(Config.manifest_copy, index=False)
save_json(leak, Config.metrics_dir / "split_report.json")

manifest rows: 4295
{'train': 3005, 'val': 645, 'test': 645}
split md5 (this run): 5fac8fa95389101fde12da21ab93388b
split md5 (VGG16)   : 5fac8fa95389101fde12da21ab93388b
manifest matches the shared split - the three models are comparable.
leakage report: {'clusters_spanning_multiple_splits': 0, 'duplicate_paths': 0, 'split_counts': {'train': 3005, 'val': 645, 'test': 645}, 'passed': True}


saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\split_report.json


In [4]:
support = df.groupby("split")[Config.class_names].sum().astype(int).loc[
    ["train", "val", "test"]]
print("images per split :", df["split"].value_counts().to_dict())
print("class support per split:")
print(support)
print("per source :", pd.crosstab(df["split"], df["source"]).to_dict())

images per split : {'train': 3005, 'val': 645, 'test': 645}
class support per split:
        D00  D10   D20   D40
split                       
train  1312  259  1320  1165
val     286   50   277   254
test    275   51   290   265
per source : {'Czech': {'test': 137, 'train': 772, 'val': 163}, 'India': {'test': 508, 'train': 2233, 'val': 482}}


## 3. Input pipeline

Images are decoded, resized to 224x224 and streamed in batches of 64.
Pixel values stay in `[0, 255]` - the model carries its own
`Rescaling(1/255)` layer, so normalisation lives inside the saved model.
No augmentation is used, matching the VGG16/ResNet-50 protocol (section 4 of
the report discusses augmentation as a shared limitation).

In [5]:
def split_xy(frame, split):
    part = frame[frame["split"] == split]
    return (list(part["path"]),
            part[Config.class_names].to_numpy(dtype=np.float32))


train_paths, y_train = split_xy(df, "train")
val_paths, y_val = split_xy(df, "val")
test_paths, y_test = split_xy(df, "test")
print(len(train_paths), len(val_paths), len(test_paths))


def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (Config.img_size, Config.img_size),
                          method="bilinear")
    return img, label


def make_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (np.asarray(paths), labels.astype(np.float32)))
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache()
    if shuffle:
        ds = ds.shuffle(1024, seed=Config.seed)
    return ds.batch(Config.batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(train_paths, y_train, shuffle=True)
val_ds = make_ds(val_paths, y_val)
test_ds = make_ds(test_paths, y_test)

sample_x, sample_y = next(iter(train_ds.take(1)))
print("batch shapes:", sample_x.shape, sample_y.shape)
print("pixel range :", float(sample_x.numpy().min()),
      "-", float(sample_x.numpy().max()))

3005 645 645


batch shapes: (64, 224, 224, 3) (64, 4)


pixel range : 0.0 - 255.0


## 4. Model architecture

Four convolutional blocks (32/64/128/256 filters, 3x3, ReLU, max-pool) are
trained end to end; a flatten layer feeds a 128-unit dense layer with dropout
0.5 and **4 sigmoid outputs** - one independent yes/no decision per damage
type, exactly like the VGG16 and ResNet-50 heads. This is the same trunk the
original country-classification experiment used, with the output layer and
loss changed to the project's damage task.

In [6]:
model = models.Sequential([
    layers.Input(shape=(Config.img_size, Config.img_size, 3)),

    # Normalisation lives inside the model (pixels enter as raw [0, 255])
    layers.Rescaling(1.0 / 255),

    # Convolution Block 1
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 2
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 3
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 4
    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Classification head
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(Config.dropout),
    layers.Dense(len(Config.class_names), activation="sigmoid"),
])

model.compile(
    optimizer=Adam(learning_rate=Config.lr),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="binary_acc"),
             tf.keras.metrics.AUC(name="auc")],
)
model.summary()

RUN["model"] = {
    "total_params": int(model.count_params()),
    "trainable_params": int(sum(tf.keras.backend.count_params(w)
                                for w in model.trainable_weights)),
    "frozen_params": int(sum(tf.keras.backend.count_params(w)
                             for w in model.non_trainable_weights)),
    "backbone": "Custom CNN (from scratch, multi-label head)",
}
print(RUN["model"])

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,718,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,107,652 (19.48 MB)

 Trainable params: 5,107,652 (19.48 MB)

 Non-trainable params: 0 (0.00 B)

{'total_params': 5107652, 'trainable_params': 5107652, 'frozen_params': 0, 'backbone': 'Custom CNN (from scratch, multi-label head)'}


## 5. Training

* `binary_crossentropy`, Adam 1e-3, metrics: binary accuracy + AUC
* `EarlyStopping(patience=12)` and `ReduceLROnPlateau(patience=5)` on
  `val_auc`, best-weights checkpoint - the same callback recipe as VGG16
* a re-run skips training when the checkpoint and `run_meta.json` match.

In [7]:
set_global_seed(Config.seed)


class BestWeights(tf.keras.callbacks.Callback):
    """Saves weights when val_auc improves."""
    def __init__(self, path, monitor="val_auc"):
        super().__init__()
        self.path = str(path)
        self.monitor = monitor
        self.best = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is not None and value > self.best:
            self.best = float(value)
            self.model.save_weights(self.path)


run_meta = {
    "split_md5": RUN["split_md5"],
    "config": Config.as_dict(),
    "model": RUN["model"],
}
train_skipped = False
if (Config.checkpoint_path.exists() and Config.weights_path.exists()
        and Config.history_path.exists() and Config.run_meta_path.exists()):
    previous = load_json(Config.run_meta_path)
    if previous == run_meta:
        train_skipped = True
        print("Valid checkpoint + matching config found -> skipping training.")

history = None
if not train_skipped:
    callbacks = [
        EarlyStopping(monitor="val_auc", mode="max",
                      patience=Config.patience_earlystop,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                          patience=Config.patience_lr, min_lr=1e-6,
                          verbose=1),
        BestWeights(Config.weights_path, monitor="val_auc"),
    ]

    with Timer() as train_timer:
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=Config.epochs,
            callbacks=callbacks,
            verbose=2,
        )
    RUN["train_seconds"] = round(train_timer.elapsed, 1)
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    save_json(hist, Config.history_path)
    save_json(run_meta, Config.run_meta_path)
    model.save(str(Config.checkpoint_path))
    print(f"trained in {RUN['train_seconds']}s, "
          f"best val AUC {max(hist['val_auc']):.4f} "
          f"(epoch {int(np.argmax(hist['val_auc'])) + 1})")
else:
    hist = load_json(Config.history_path)
    RUN["train_seconds"] = None
    model.load_weights(str(Config.weights_path))
    print("loaded history:", {k: round(v[-1], 4) for k, v in hist.items()})

Epoch 1/100


C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


47/47 - 162s - 3s/step - auc: 0.7114 - binary_acc: 0.6771 - loss: 0.5713 - val_auc: 0.7759 - val_binary_acc: 0.7081 - val_loss: 0.5325 - learning_rate: 0.0010


Epoch 2/100


47/47 - 150s - 3s/step - auc: 0.7546 - binary_acc: 0.6993 - loss: 0.5377 - val_auc: 0.7833 - val_binary_acc: 0.7143 - val_loss: 0.5138 - learning_rate: 0.0010


...

47/47 - 58s - 1s/step - auc: 0.9734 - binary_acc: 0.9065 - loss: 0.1952 - val_auc: 0.7995 - val_binary_acc: 0.7426 - val_loss: 0.7873 - learning_rate: 2.5000e-04


Epoch 21: early stopping


Restoring model weights from the end of the best epoch: 9.


saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\training_history.json
saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\run_meta.json


trained in 1576.4s, best val AUC 0.8205 (epoch 9)


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("Binary cross-entropy")
axes[0].legend()
axes[1].plot(hist["binary_acc"], label="train")
axes[1].plot(hist["val_binary_acc"], label="val")
axes[1].set_title("Binary accuracy")
axes[1].legend()
axes[2].plot(hist["auc"], label="train")
axes[2].plot(hist["val_auc"], label="val")
axes[2].set_title("AUC")
axes[2].legend()
plt.tight_layout()
plt.savefig(Config.fig_dir / "learning_curves.png", dpi=150)
plt.close()

best_ep = int(np.argmax(hist["val_auc"])) + 1
print(f"epochs run       : {len(hist['loss'])}")
print(f"best epoch       : {best_ep} (val AUC {max(hist['val_auc']):.4f})")
print(f"final val loss   : {hist['val_loss'][-1]:.4f}")

epochs run       : 21
best epoch       : 9 (val AUC 0.8205)
final val loss   : 0.7873


## 6. Evaluation on the test set (threshold 0.5)

The test set is loaded and scored only here. Metrics: subset accuracy,
hamming accuracy, per-class precision / recall / F1 / ROC-AUC, micro and
macro averages - the exact suite the other damage models report.

In [9]:
model.load_weights(str(Config.weights_path))
print("loaded best weights:", Config.weights_path.name)

prob_val = model.predict(val_ds, verbose=0)
prob_test = model.predict(test_ds, verbose=0)
pred_val = (prob_val >= 0.5).astype(int)
pred_test = (prob_test >= 0.5).astype(int)

assert prob_test.shape == y_test.shape


def multilabel_metrics(y_true, pred, prob, average):
    return {
        "precision": float(precision_score(y_true, pred, average=average,
                                            zero_division=0)),
        "recall": float(recall_score(y_true, pred, average=average,
                                     zero_division=0)),
        "f1": float(f1_score(y_true, pred, average=average, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, prob, average=average)),
    }


metrics = {
    "threshold": 0.5,
    "subset_accuracy": float(accuracy_score(y_test, pred_test)),
    "hamming_accuracy": float((y_test == pred_test).mean()),
    "micro": multilabel_metrics(y_test, pred_test, prob_test, "micro"),
    "macro": multilabel_metrics(y_test, pred_test, prob_test, "macro"),
    "per_class": {},
}
for i, c in enumerate(Config.class_names):
    metrics["per_class"][c] = {
        "precision": float(precision_score(y_test[:, i], pred_test[:, i],
                                           zero_division=0)),
        "recall": float(recall_score(y_test[:, i], pred_test[:, i],
                                     zero_division=0)),
        "f1": float(f1_score(y_test[:, i], pred_test[:, i], zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test[:, i], prob_test[:, i])),
        "support": int(y_test[:, i].sum()),
    }
save_json(metrics, Config.metrics_dir / "test_metrics.json")

print(f"subset accuracy : {metrics['subset_accuracy']:.4f}")
print(f"hamming accuracy: {metrics['hamming_accuracy']:.4f}")
print(f"micro  P/R/F1   : {metrics['micro']['precision']:.3f} / "
      f"{metrics['micro']['recall']:.3f} / {metrics['micro']['f1']:.3f}")
print(f"macro  P/R/F1   : {metrics['macro']['precision']:.3f} / "
      f"{metrics['macro']['recall']:.3f} / {metrics['macro']['f1']:.3f}")
print(f"micro/macro AUC : {metrics['micro']['roc_auc']:.3f} / "
      f"{metrics['macro']['roc_auc']:.3f}")
pd.DataFrame(metrics["per_class"]).T

loaded best weights: custom_cnn_weights.weights.h5


saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\test_metrics.json
subset accuracy : 0.3612
hamming accuracy: 0.7353
micro  P/R/F1   : 0.646 / 0.496 / 0.561
macro  P/R/F1   : 0.578 / 0.397 / 0.435
micro/macro AUC : 0.804 / 0.760


,precision,recall,f1,roc_auc,support
D00,0.723077,0.341818,0.464198,0.685926,275.0
D10,0.333333,0.019608,0.037037,0.853964,51.0
D20,0.643564,0.672414,0.657673,0.785158,290.0
D40,0.612500,0.554717,0.582178,0.715204,265.0


In [10]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for i, (c, ax) in enumerate(zip(Config.class_names, axes)):
    fpr, tpr, _ = roc_curve(y_test[:, i], prob_test[:, i])
    ax.plot(fpr, tpr, lw=1.8,
            label=f"{c} (AUC {metrics['per_class'][c]['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.set_title(f"ROC - {c}")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(Config.fig_dir / "roc_curves.png", dpi=150)
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for i, (c, ax) in enumerate(zip(Config.class_names, axes.ravel())):
    cm = confusion_matrix(y_test[:, i], pred_test[:, i])
    im = ax.imshow(cm, cmap="Blues")
    for (r0, c0), val in np.ndenumerate(cm):
        ax.text(c0, r0, str(val), ha="center", va="center",
                color="black" if val < cm.max() / 2 else "white")
    ax.set_title(c)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
plt.colorbar(im, ax=axes, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(Config.fig_dir / "confusion_matrices.png", dpi=150)
plt.close()
print("saved roc_curves.png + confusion_matrices.png")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_41544\2228199465.py:28: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


saved roc_curves.png + confusion_matrices.png


## 7. Threshold tuning (validation only)

0.5 is rarely optimal for imbalanced labels. For each class we sweep
thresholds on the **validation** set and keep the value with the best F1;
the tuned predictions are then evaluated once on the test set.

In [11]:
grid = np.arange(0.05, 0.95, 0.05)
best_thr = {}
for i, c in enumerate(Config.class_names):
    f1s = [f1_score(y_val[:, i], (prob_val[:, i] >= t).astype(int),
                    zero_division=0) for t in grid]
    best_thr[c] = float(grid[int(np.argmax(f1s))])
print("tuned thresholds (from validation):", best_thr)

pred_test_tuned = np.stack(
    [(prob_test[:, i] >= best_thr[c]).astype(int)
     for i, c in enumerate(Config.class_names)], axis=1)

metrics_tuned = {
    "thresholds": best_thr,
    "subset_accuracy": float(accuracy_score(y_test, pred_test_tuned)),
    "micro": multilabel_metrics(y_test, pred_test_tuned, prob_test, "micro"),
    "macro": multilabel_metrics(y_test, pred_test_tuned, prob_test, "macro"),
    "per_class": {},
}
for i, c in enumerate(Config.class_names):
    metrics_tuned["per_class"][c] = {
        "precision": float(precision_score(y_test[:, i],
                                           pred_test_tuned[:, i],
                                           zero_division=0)),
        "recall": float(recall_score(y_test[:, i], pred_test_tuned[:, i],
                                     zero_division=0)),
        "f1": float(f1_score(y_test[:, i], pred_test_tuned[:, i],
                             zero_division=0)),
        "roc_auc": metrics["per_class"][c]["roc_auc"],
        "support": int(y_test[:, i].sum()),
    }
save_json(metrics_tuned, Config.metrics_dir / "test_metrics_tuned.json")
save_json(best_thr, Config.metrics_dir / "thresholds.json")

cmp_df = pd.DataFrame({
    "F1 @0.5": {c: metrics["per_class"][c]["f1"] for c in Config.class_names},
    "F1 tuned": {c: metrics_tuned["per_class"][c]["f1"]
                 for c in Config.class_names},
    "threshold": best_thr,
})
cmp_df.loc["micro"] = [metrics["micro"]["f1"], metrics_tuned["micro"]["f1"],
                       np.nan]
cmp_df.loc["macro"] = [metrics["macro"]["f1"], metrics_tuned["macro"]["f1"],
                       np.nan]
print(f"subset accuracy 0.5 -> tuned: "
      f"{metrics['subset_accuracy']:.4f} -> "
      f"{metrics_tuned['subset_accuracy']:.4f}")
cmp_df

tuned thresholds (from validation): {'D00': 0.3, 'D10': 0.15000000000000002, 'D20': 0.45, 'D40': 0.45}
saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\test_metrics_tuned.json
saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\thresholds.json
subset accuracy 0.5 -> tuned: 0.3612 -> 0.2217


,F1 @0.5,F1 tuned,threshold
D00,0.464198,0.619835,0.30
D10,0.037037,0.457516,0.15
D20,0.657673,0.724812,0.45
D40,0.582178,0.623468,0.45
micro,0.561336,0.642080,NaN
macro,0.435271,0.606408,NaN


In [12]:
rows = []
test_df = df[df["split"] == "test"].reset_index(drop=True)
for source, grp in test_df.groupby("source"):
    idx = grp.index.to_numpy()
    for name, pred in (("0.5", pred_test), ("tuned", pred_test_tuned)):
        rows.append({
            "source": source,
            "threshold": name,
            "n": int(len(idx)),
            "micro_f1": float(f1_score(y_test[idx], pred[idx],
                                       average="micro", zero_division=0)),
            "subset_acc": float(accuracy_score(y_test[idx], pred[idx])),
        })
source_df = pd.DataFrame(rows)
save_json(source_df.to_dict("records"),
          Config.metrics_dir / "source_metrics.json")
source_df

saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\source_metrics.json


,source,threshold,n,micro_f1,subset_acc
0,Czech,0.5,137,0.637874,0.481752
1,Czech,tuned,137,0.655340,0.233577
2,India,0.5,508,0.542994,0.328740
3,India,tuned,508,0.638873,0.218504


## 8. Computational considerations

Wall-clock cost, parameter budget and environment - the honest account that
accompanies every performance table in the report.

In [13]:
import os as _os

RUN["computational"] = {
    **RUN.get("model", {}),
    "feature_extraction_seconds": None,
    "training_seconds": RUN.get("train_seconds"),
    "epochs_run": len(hist["loss"]),
    "best_epoch": int(np.argmax(hist["val_auc"])) + 1,
    "checkpoint_mb": round(_os.path.getsize(Config.checkpoint_path) / 1e6, 2),
    "backbone": "Custom CNN (from scratch)",
    "device": "CPU (TensorFlow 2.21, native Windows - no GPU support)",
}
RUN["dataset"] = {
    "labelled_images": int(len(df)),
    "train": len(train_paths),
    "val": len(val_paths),
    "test": len(test_paths),
}
save_json(RUN, Config.metrics_dir / "run_summary.json")
print(json.dumps(RUN["computational"], indent=2))

saved -> C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\metrics\run_summary.json
{
  "total_params": 5107652,
  "trainable_params": 5107652,
  "frozen_params": 0,
  "backbone": "Custom CNN (from scratch)",
  "feature_extraction_seconds": null,
  "training_seconds": 1576.4,
  "epochs_run": 21,
  "best_epoch": 9,
  "checkpoint_mb": 61.35,
  "device": "CPU (TensorFlow 2.21, native Windows - no GPU support)"
}


## 9. Reproducibility and status

* seed 42 (Python / NumPy / TensorFlow), config in `run_meta.json`
* shared split manifest, MD5 recorded in `run_summary.json`
* artefacts: `results/IT23331518/metrics/*.json`,
  `results/IT23331518/figures/*.png`, `custom_cnn_model.keras`
* test set used only in section 6; thresholds tuned on validation only

In [14]:
print("=" * 60)
print("CUSTOM CNN - FINAL DAMAGE RESULTS")
print("=" * 60)
print(f"split md5        : {RUN['split_md5']}")
print(f"micro F1  @0.5   : {metrics['micro']['f1']:.4f}  "
      f"(tuned {metrics_tuned['micro']['f1']:.4f})")
print(f"macro F1  @0.5   : {metrics['macro']['f1']:.4f}  "
      f"(tuned {metrics_tuned['macro']['f1']:.4f})")
print(f"micro AUC        : {metrics['micro']['roc_auc']:.4f}")
print(f"subset accuracy  : {metrics['subset_accuracy']:.4f}  "
      f"(tuned {metrics_tuned['subset_accuracy']:.4f})")
print(f"epochs / best    : {RUN['computational']['epochs_run']} / "
      f"{RUN['computational']['best_epoch']}")
print(f"model            : {Config.checkpoint_path} "
      f"({RUN['computational']['checkpoint_mb']} MB)")

CUSTOM CNN - FINAL DAMAGE RESULTS
split md5        : 5fac8fa95389101fde12da21ab93388b
micro F1  @0.5   : 0.5613  (tuned 0.6421)
macro F1  @0.5   : 0.4353  (tuned 0.6064)
micro AUC        : 0.8043
subset accuracy  : 0.3612  (tuned 0.2217)
epochs / best    : 21 / 9
model            : C:\Users\ASUS\Desktop\SLIIT\Year 4\Semester 2\Deep Learning\Assignment\SE4050-Deep-Learning-Assignment\results\IT23331518\custom_cnn_model.keras (61.35 MB)
